# Dependencias

In [ ]:
import ctypes
ctypes.CDLL("/home/gcamilo/envs/wrf312/lib/libgomp.so.1", mode=ctypes.RTLD_GLOBAL)
import sys
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
import glob
import random
import pandas as pd
import copy
import re
import wrf
import cartopy.feature as cfeature
import cartopy.crs as crs
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from tqdm.auto import tqdm
from datetime import datetime
from IPython.display import display
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from skimage.metrics import structural_similarity as ssim
from wrf import getvar, get_cartopy, latlon_coords, to_np
from netCDF4 import Dataset as ncDataset
from scipy.ndimage import gaussian_filter1d

ruta_padre = os.path.abspath('..')
if ruta_padre not in sys.path:
    sys.path.append(ruta_padre)

CROP     = 1
# Rutas a las dos carpetas de salidas WRF (separadas por espacio en disco)
dir_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
dir_2019 = "../salidasWRF2019"
archivos = sorted(
    glob.glob(os.path.join(dir_2018, "wrfout_d01_*")) +
    glob.glob(os.path.join(dir_2019, "wrfout_d01_*"))
)
with ncDataset(archivos[0]) as ds0:
    ref = getvar(ds0, "LIGHTDENS")
    CART_PROJ = get_cartopy(ref)
    LATS, LONS = latlon_coords(ref)
LATS, LONS = to_np(LATS)[CROP:-CROP, CROP:-CROP], to_np(LONS)[CROP:-CROP, CROP:-CROP]

from modulos.tormentaDataset import TormentaDataset

# Chequeamos si hay GPU disponible para acelerar el entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
print("=== DIAGNÓSTICO DE PYTORCH Y CUDA ===")
print("Versión de PyTorch:", torch.__version__)
print("Versión de CUDA instalada en PyTorch:", torch.version.cuda)
print("¿CUDA disponible?:", torch.cuda.is_available())
print("Cantidad de GPUs detectadas:", torch.cuda.device_count())
print("=====================================\n")

if torch.cuda.is_available():
    try:
        device = torch.device("cuda")
        # Forzamos una operación matemática en la placa para validar el puente físico
        torch.zeros(1).to(device)
        print("✅ ¡Conexión exitosa a la GPU!")
        print(f"Propiedades del hardware asignado: {torch.cuda.get_device_name(0)}")
    except RuntimeError as e:
        print("❌ Error al interactuar con el hardware:", e)
else:
    print("⚠️ CUDA no está disponible en absoluto para este binario de PyTorch.")

In [ ]:
# Para REPRODUCIBILIDAD

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Ext U-Net Comb V2.1

(Se llamó V2.2 durante el desarrollo; las variantes de la lista son las de la numeración interna.)

- 2.1: Coeficientes de las funciones de pérdida agregados
- 2.2: Utiliza valor real de LIGHTDENS en WMSE
- 2.3: Combina 2.1 y 2.2

In [ ]:
class UNet_WRF(nn.Module):
    """
    U-Net con profundidad y ancho configurables.

    Parámetros
    ----------
    in_channels : int, default 6
        Cantidad de canales de entrada (variables WRF).
    n_niveles : int, default 2
        Cantidad de niveles de encoder/decoder (pooling + upsampling).
        La arquitectura original tenía 2 niveles.
    n_filtros_base : int, default 32
        Cantidad de filtros en el primer nivel del encoder. Cada nivel
        siguiente duplica esta cantidad (32→64→128→...), igual que el
        bottleneck, que duplica una vez más el último nivel.
    kernel_sizes : list[int] o None
        Un kernel_size por nivel del encoder (y su espejo en el decoder).
        Si es None, usa kernel_size=3 en todos los niveles.
        Debe tener longitud == n_niveles.
    dropout_ps : list[float] o None
        Un dropout_p por nivel del encoder (y su espejo en el decoder),
        aplicado también en el bottleneck (usa el último valor de la lista).
        Si es None, no aplica dropout en ningún nivel (comportamiento original).
        Debe tener longitud == n_niveles.
    input_size : int, default 297
        Tamaño espacial (H=W) de la imagen de entrada, usado para calcular
        el padding necesario según la profundidad elegida.
    """
    def __init__(self, in_channels=6, n_niveles=2, n_filtros_base=32,
                 kernel_sizes=None, dropout_ps=None, input_size=297):
        super(UNet_WRF, self).__init__()

        if kernel_sizes is None:
            kernel_sizes = [3] * n_niveles
        assert len(kernel_sizes) == n_niveles, \
            f"kernel_sizes debe tener {n_niveles} elementos, tiene {len(kernel_sizes)}"

        if dropout_ps is None:
            dropout_ps = [0.0] * n_niveles
        assert len(dropout_ps) == n_niveles, \
            f"dropout_ps debe tener {n_niveles} elementos, tiene {len(dropout_ps)}"
        
        self.n_niveles = n_niveles
        self.dropout_ps = dropout_ps

        # --- Padding espacial dinámico según profundidad ---
        divisor = 2 ** n_niveles
        size_ajustado = ((input_size // divisor) + 1) * divisor
        pad_total = size_ajustado - input_size
        # Repartimos el padding: menos a la izquierda/arriba, más a la derecha/abajo
        self.pad_izq_arr = pad_total // 2
        self.pad_der_aba = pad_total - self.pad_izq_arr
        self.input_size = input_size

        # --- Canales por nivel
        canales = [n_filtros_base * (2 ** i) for i in range(n_niveles)]
        canal_bottleneck = n_filtros_base * (2 ** n_niveles)
        # Kernel del bottleneck: reusamos el del nivel más profundo
        k_bottleneck = kernel_sizes[-1]
        dropout_bottleneck = dropout_ps[-1]

        def bloque_conv(canal_in, canal_out, k, dropout_p):
            pad = k // 2
            capas = [
                nn.Conv2d(canal_in, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
                nn.Conv2d(canal_out, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
            ]
            if dropout_p > 0:
                capas.append(nn.Dropout2d(p=dropout_p))
            return nn.Sequential(*capas)
        
        # --- ENCODER (dinámico) ---
        self.enc_blocks = nn.ModuleList()
        self.pools      = nn.ModuleList()
        canal_in = in_channels
        for canal_out, k, p in zip(canales, kernel_sizes, dropout_ps):
            self.enc_blocks.append(bloque_conv(canal_in, canal_out, k, p))
            self.pools.append(nn.MaxPool2d(2, 2))
            canal_in = canal_out

        # --- BOTTLENECK ---
        self.bottleneck = bloque_conv(canal_in, canal_bottleneck, k_bottleneck, dropout_bottleneck)

        # --- DECODER (dinámico, en orden inverso al encoder) ---
        self.ups        = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        canal_in = canal_bottleneck
        for canal_skip, k, p in zip(reversed(canales), reversed(kernel_sizes), reversed(dropout_ps)):
            self.ups.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
            self.dec_blocks.append(bloque_conv(canal_in + canal_skip, canal_skip, k, p))
            canal_in = canal_skip

        # --- SALIDA ---
        self.salida  = nn.Conv2d(canal_in, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()  # SOLO para FACL

    def forward(self, x):
        x_pad = F.pad(x, (self.pad_izq_arr, self.pad_der_aba,
                           self.pad_izq_arr, self.pad_der_aba))

        # --- Encoder ---
        skips = []
        h = x_pad
        for bloque, pool in zip(self.enc_blocks, self.pools):
            h = bloque(h)
            skips.append(h)       # guardamos para el skip connection
            h = pool(h)

        # --- Bottleneck ---
        h = self.bottleneck(h)

        # --- Decoder ---
        for up, bloque, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            h = up(h)
            h = torch.cat([h, skip], dim=1)
            h = bloque(h)

        out = self.salida(h)
        out = self.sigmoid(out)  # SOLO para FACL

        # Recortar el padding agregado al principio
        out = out[:, :, self.pad_izq_arr:self.pad_izq_arr + self.input_size,
                         self.pad_izq_arr:self.pad_izq_arr + self.input_size]
        return out

In [ ]:
# HP

modelo_unet = UNet_WRF(
    in_channels=3, 
    n_niveles=3, 
    n_filtros_base=64,
    kernel_sizes=[5, 3, 3], 
    dropout_ps=[0.0, 0.0, 0.0]
).to(device)

In [ ]:
print("\n\n=== RESUMEN DE LA U-NET ===")
summary(modelo_unet, input_size=(1, 3, 297, 297), 
        col_names=["input_size", "output_size", "num_params"])

# Entrenamiento

In [ ]:
# ==========================================
# FUNCION DE PÉRDIDA
# ==========================================
class HybridFourierLossV2(nn.Module):
    """
    Variante de FACL con tres ramas:
        - WMSE  (seleccionada con probabilidad creciente P(t))
        - FAL   (seleccionada con prob (1-P(t))/2 cuando no se elige WMSE)
        - FCL   (seleccionada con prob (1-P(t))/2 cuando no se elige WMSE)

    Conforme avanza el entrenamiento, el modelo tiende a optimizar
    WMSE, mientras FAL y FCL se usan cada vez menos.
    En validación se usa una mezcla fija 1/3 de cada una.

    Parámetros
    ----------
    min_prob, max_prob, prob_slope : float
        Controlan la transición hacia WMSE (igual que RandomFourierLoss).
    peso_positivo : float
        Peso de los píxeles activos en WMSE. Default 20.
    min_y, max_y : float
        Estadísticos de normalización del target, necesarios para
        identificar píxeles activos dentro de WMSE.
    """
    def __init__(self, min_prob=0.0, max_prob=0.7, prob_slope=1e-5,
                 peso_positivo=20.0, min_y=0.0, max_y=1.0):
        super(HybridFourierLossV2, self).__init__()
        self.min_prob      = min_prob
        self.max_prob      = max_prob
        self.prob_slope    = prob_slope
        self.peso_positivo = peso_positivo
        self.step          = 0
        self.min_y         = min_y
        self.max_y         = max_y
        # V2.1 ONLY
        # Empezamos en 0 → sigma=1 → sin sesgo inicial entre términos — se aprenden junto con la red.
        # self.log_var_wmse = nn.Parameter(torch.tensor(8.0))
        # self.log_var_fal  = nn.Parameter(torch.tensor(0.0))
        # self.log_var_fcl  = nn.Parameter(torch.tensor(0.0))

    # V2.1 ONLY
    # def _ponderar(self, loss_val, log_var):
    #    """
    #    Aplica Uncertainty Weighting a un término de pérdida individual.
    #    
    #    Parámetros
    #    ----------
    #    loss_val : Tensor escalar
    #        El valor escalar de la función de pérdida original calculada 
    #        (por ejemplo, el resultado bruto de WMSE, FAL o FCL).
    #    
    #    log_var : nn.Parameter (Tensor escalar)
    #        El logaritmo de la varianza asociado a este término de pérdida. 
    #        Es un parámetro entrenable de la red que indica qué tan 
    #        "ruidosa" o incierta es esta tarea.
    #
    #    Returns
    #    -------
    #    Tensor escalar
    #        El valor de la pérdida ponderada.
    #    """
    #    precision = torch.exp(-log_var)
    #    return precision * loss_val + log_var
    
    def fcl(self, fft_pred, fft_truth, eps=1e-8):
        """
        Fourier Correlation Loss (FCL).

        Mide la correlación normalizada entre las transformadas de Fourier
        de la predicción y del ground truth.

        Matemáticamente, para cada imagen i del batch:

            FCL_i = 1 - Re(<F_pred_i, conj(F_truth_i)>) /
                        (||F_pred_i|| · ||F_truth_i||)

        donde Re(·) extrae la parte real del producto interno complejo.
        El valor final es el promedio de FCL_i sobre todas las imágenes del batch.

        Esta pérdida depende de la estructura global de la imagen
        (fase + organización espacial), pero es prácticamente
        invariante ante cambios globales de intensidad.

        Parámetros
        ----------
        fft_pred : Tensor complejo
            FFT de la predicción.

        fft_truth : Tensor complejo
            FFT del ground truth.

        eps : float, default 1e-8
            Constante de estabilidad numérica para evitar división por cero.

        Returns
        -------
        Tensor escalar
            Valor promedio de la Fourier Correlation Loss sobre el batch.
        """
        dims        = tuple(range(1, fft_pred.dim()))
        numerador   = torch.sum((fft_pred * torch.conj(fft_truth)).real, dim=dims)
        norm_pred   = torch.sqrt(torch.sum(fft_pred.abs() ** 2, dim=dims))
        norm_truth  = torch.sqrt(torch.sum(fft_truth.abs() ** 2, dim=dims))
        correlacion = numerador / (norm_pred * norm_truth + eps)
        return (1 - correlacion).mean()

    def fal(self, fft_pred, fft_truth):
        """
        Fourier Amplitude Loss (FAL).

        Compara únicamente el espectro de amplitudes de ambas imágenes,
        ignorando completamente la fase.

        Matemáticamente:

            FAL = mean((|F_pred| - |F_gt|)^2)

        Esta pérdida favorece que la distribución de frecuencias sea la
        correcta, produciendo imágenes más nítidas. Sin embargo, por sí
        sola no puede reconstruir correctamente la estructura espacial,
        ya que la fase del ground truth no interviene en el gradiente.

        Parámetros
        ----------
        fft_pred : Tensor complejo
            FFT de la predicción.

        fft_truth : Tensor complejo
            FFT del ground truth.

        Returns
        -------
        Tensor escalar
            Valor de la Fourier Amplitude Loss.
        """
        return ((fft_pred.abs() - fft_truth.abs()) ** 2).mean()

    def wmse(self, pred, gt):
        """
        MSE ponderado: penaliza con mayor peso los píxeles donde hay actividad
        eléctrica real (y_real > 0), para compensar el fuerte desbalance espacial.
        Tanto pred como gt se reciben en escala normalizada [0,1]; el error se
        calcula en escala real de LIGHTDENS (flash/columna).
        """
        pred_log  = pred * (self.max_y - self.min_y) + self.min_y
        pred_real = torch.clamp(torch.exp(pred_log) - 0.1, min=0)
    
        # Desnormalizamos gt completamente para identificar píxeles activos
        gt_log  = gt * (self.max_y - self.min_y) + self.min_y  # [0,1] → log(y+0.1)
        gt_real = torch.clamp(torch.exp(gt_log) - 0.1, min=0)  # log → LIGHTDENS real

        diferencia = (pred_real - gt_real)**2
        pesos = torch.where(gt_real > 0,
                            torch.tensor(self.peso_positivo, device=gt.device),
                            torch.tensor(1.0, device=gt.device))
        return torch.mean(diferencia * pesos)

    def forward(self, pred, gt, mode='train'):
        pred_sq = pred.squeeze(1)   # (N,1,H,W) → (N,H,W)
        gt_sq   = gt.squeeze(1)

        fft_pred = torch.fft.fftn(pred_sq, dim=[-1, -2], norm='ortho')
        fft_gt   = torch.fft.fftn(gt_sq,   dim=[-1, -2], norm='ortho')

        H, W   = pred_sq.shape[-2:]
        weight = np.sqrt(H * W)

        prob_t = (self.step * self.prob_slope) * (self.max_prob - self.min_prob) + self.min_prob
        prob_t = min(prob_t, self.max_prob)

        if mode == 'val':
            l_wmse = self.wmse(pred, gt)
            l_fal  = self.fal(fft_pred, fft_gt) * weight
            l_fcl  = self.fcl(fft_pred, fft_gt) * weight
            # V2.1 ONLY
            # loss = (self._ponderar(l_wmse, self.log_var_wmse) +
            #        self._ponderar(l_fal,  self.log_var_fal)  +
            #        self._ponderar(l_fcl,  self.log_var_fcl)) / 3.0
            loss = (l_wmse + l_fal + l_fcl) / 3.0
            
        else:
            if np.random.rand() < prob_t:
                # V2.1 ONLY
                # loss = self._ponderar(self.wmse(pred, gt), self.log_var_wmse)
                 loss = self.wmse(pred, gt)
            else:
                if np.random.rand() < 0.5:
                    # V2.1 ONLY
                    # loss = self._ponderar(self.fal(fft_pred, fft_gt) * weight, self.log_var_fal)
                    loss = self.fal(fft_pred, fft_gt) * weight
                else:
                    # V2.1 ONLY
                    # loss = self._ponderar(self.fcl(fft_pred, fft_gt) * weight, self.log_var_fcl)
                    loss = self.fcl(fft_pred, fft_gt) * weight
            self.step += 1
                
        return loss

In [ ]:
# =====================================================================
# DATASETS
# =====================================================================

ruta_train = '../dataset/train'
ruta_val   = '../dataset/val'
ruta_stats = '../dataset/stats.npz'

print("Mapeando archivos y cargando estadísticas globales...")
train_dataset = TormentaDataset(split_dir=ruta_train, stats_path=ruta_stats)
val_dataset   = TormentaDataset(split_dir=ruta_val,   stats_path=ruta_stats)
print(f"-> Train: {len(train_dataset)} muestras | Val: {len(val_dataset)} muestras")
print(f"-> min_y: {train_dataset.min_y:.4f} | max_y: {train_dataset.max_y:.4f}")

vacias = 0
for _, y in DataLoader(train_dataset, batch_size=64, shuffle=False):
    vacias += (y.flatten(1).amax(dim=1) < 1e-6).sum().item()
print(f"-> Targets vacíos en train: {vacias} / {len(train_dataset)} ({100*vacias/len(train_dataset):.2f}%)")

In [ ]:
# =====================================================================
# HIPERPARÁMETROS
# =====================================================================

# --- Modelo ---
NOMBRE_MODELO = 'Fat Short Unet Comb V2.2'

# --- Optimización ---
LEARNING_RATE = 2.5e-4
EPOCHS        = 200
PATIENCE      = 20
BATCH_SIZE    = 16
MAX_NORM      = 1000

# --- HybridFourierLossV2 ---
MIN_PROB             = 0.0
MAX_PROB             = 0.7
EPOCAS_PARA_MAX_PROB = 60
PESO_POSITIVO        = 17.53

# ==========================================
# DATALOADERS
# ==========================================
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"-> Batches por época (Train): {len(train_loader)}")
print(f"-> Batches por época (Val):   {len(val_loader)}")

PROB_SLOPE = (MAX_PROB - MIN_PROB) / (EPOCAS_PARA_MAX_PROB * len(train_loader))

# --- Instanciación ---
modelo = UNet_WRF(
    in_channels=3, 
    n_niveles=3, 
    n_filtros_base=64,
    kernel_sizes=[5, 3, 3], 
    dropout_ps=[0.0, 0.0, 0.0]
).to(device)

criterion = HybridFourierLossV2(
    min_prob=MIN_PROB, max_prob=MAX_PROB, prob_slope=PROB_SLOPE,
    peso_positivo=PESO_POSITIVO,
    min_y=train_dataset.min_y, max_y=train_dataset.max_y
).to(device)

optimizer = optim.Adam(
    # V2.1 ONLY
    # list(modelo.parameters()) + list(criterion.parameters()),
    modelo.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-7
)

# --- Historiales ---
historial_train_loss   = []
historial_val_loss     = []
historial_tiempo_epoch = []

print(f"-> Modelo:         {NOMBRE_MODELO}")
print(f"-> Corriendo en:   {device}")
print(f"-> Batch Size:     {BATCH_SIZE}")
print(f"-> Épocas totales: {EPOCHS}")
print(f"-> Patience (ES):  {PATIENCE} épocas")
print(f"-> Learning Rate:  {LEARNING_RATE:.6f}")
print(f"-> Grad Clipping:  {MAX_NORM:.2f} (max_norm)")
print(f"-> Función Loss:   HybridFourierLoss (min_prob={MIN_PROB}, max_prob={MAX_PROB:.4f}, peso_positivo={PESO_POSITIVO:.4f})")
print(f"-> Épocas para Max Prob: {EPOCAS_PARA_MAX_PROB}")

In [ ]:
# ==========================================
# BUCLE DE ENTRENAMIENTO
# ==========================================
CARPETA_MODELOS = '../modelos'
mejor_val_loss = float('inf')
epocas_sin_mejora = 0
mejor_pesos = None
mejor_epoca = 0

os.makedirs(CARPETA_MODELOS, exist_ok=True)
fecha = datetime.now().strftime("%Y%m%d_%H%M")
ruta_guardado = os.path.join(CARPETA_MODELOS, f"{fecha}_{NOMBRE_MODELO}.pth")

print(f"--- INICIANDO ENTRENAMIENTO (máx. {EPOCHS} épocas, patience={PATIENCE}) ---")
tiempo_inicio_total = time.time()
torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # ==========================================
    # FASE DE ENTRENAMIENTO
    # ==========================================
    modelo.train() 
    train_loss_acumulada = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Train]", leave=False)
    
    for batch_X, batch_y in train_pbar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()                         
        predicciones = modelo(batch_X)         
        loss = criterion(predicciones, batch_y, mode='train')       
        loss.backward()    
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=MAX_NORM)
        optimizer.step()                              
        
        train_loss_acumulada += loss.item() * batch_X.size(0)
        
        # Actualizamos el texto a la derecha de la barra para ver el loss en tiempo real
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_train_loss = train_loss_acumulada / len(train_loader.dataset)
    historial_train_loss.append(epoch_train_loss)
    
    # ==========================================
    # FASE DE VALIDACIÓN
    # ==========================================
    modelo.eval()
    val_loss_acumulada = 0.0
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Val]  ", leave=False)
    
    with torch.no_grad(): 
        for batch_X, batch_y in val_pbar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            predicciones = modelo(batch_X)
            loss = criterion(predicciones, batch_y, mode='val')
            
            val_loss_acumulada += loss.item() * batch_X.size(0)
            val_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    epoch_val_loss = val_loss_acumulada / len(val_loader.dataset)
    historial_val_loss.append(epoch_val_loss)
    tiempo_epoch = time.time() - start_time
    historial_tiempo_epoch.append(tiempo_epoch)

    # ==========================================
    # EARLY STOPPING
    # ==========================================
    if epoch_val_loss < mejor_val_loss:
        mejor_val_loss = epoch_val_loss
        epocas_sin_mejora = 0
        mejor_epoca = epoch + 1
        # Guardamos una copia en memoria de los mejores pesos
        mejor_pesos = copy.deepcopy(modelo.state_dict())
    else:
        epocas_sin_mejora += 1
    
    # ==========================================
    # GUARDADO
    # ==========================================
    torch.save({
        'model_state_dict': modelo.state_dict(),
        'best_model_state_dict': mejor_pesos,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict':  scheduler.state_dict(),
        # V2.1 ONLY
        # 'criterion_state_dict': criterion.state_dict(),
        'historial_train_loss': historial_train_loss,
        'historial_val_loss': historial_val_loss,
        'historial_tiempo_epoch': historial_tiempo_epoch,
        'mejor_epoca': mejor_epoca,
        'mejor_val_loss': mejor_val_loss,
        'epochs_entrenadas': epoch + 1,
        'lr': LEARNING_RATE,
        'patience': PATIENCE,
    }, ruta_guardado)

    # ==========================================
    # SCHEDULER STEP
    # ==========================================
    scheduler.step()
    lr_actual = optimizer.param_groups[0]['lr']
    
    # ==========================================
    # REPORTE DE LA ÉPOCA
    # ==========================================
    tiempo_epoch = time.time() - start_time
    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Tiempo: {tiempo_epoch:.2f}s | "
          f"Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f} | "
          f"Sin mejora: {epocas_sin_mejora}/{PATIENCE}")
    
    # ==========================================
    # CORTE POR EARLY STOPPING
    # ==========================================
    if epocas_sin_mejora >= PATIENCE:
        print(f"\nEarly stopping activado en la época {epoch+1}.")
        print(f"Mejor val_loss: {mejor_val_loss:.6f} (época {mejor_epoca})")
        break

# ==========================================
# RESTAURAR MEJORES PESOS
# ==========================================
modelo.load_state_dict(mejor_pesos)
print(f"\nPesos restaurados a la mejor época ({mejor_epoca}).")

# ==========================================
# REPORTE FINAL
# ==========================================
tiempo_fin_total = time.time()
segundos_totales = tiempo_fin_total - tiempo_inicio_total
horas_totales = segundos_totales / 3600
horas_enteras = int(horas_totales)
minutos_enteros = int((segundos_totales % 3600) / 60)

print("\n¡Entrenamiento finalizado exitosamente!")
print(f"Modelo final disponible en: {ruta_guardado}")
print(f"Tiempo total: {horas_totales:.2f} horas ({horas_enteras}h {minutos_enteros}m)")
print(f"Tiempo promedio por época: {np.mean(historial_tiempo_epoch):.1f}s")
print(f"Mejor época: {mejor_epoca} | Mejor val_loss: {mejor_val_loss:.6f}")

## Diagnóstico de norma de gradiente
En base a los resultados, se decide elegir `max_norm=1000.0` con la finalidad de minimizar la norma para los valores outliers

In [ ]:
# ==========================================
# DIAGNÓSTICO: NORMA DEL GRADIENTE (sin clipping real)
# ==========================================
normas_gradiente = []

modelo.train()
diag_pbar = tqdm(train_loader, desc="Diagnóstico de norma de gradiente", leave=True)

for batch_X, batch_y in diag_pbar:
    batch_X, batch_y = batch_X.to(device), batch_y.to(device)

    optimizer.zero_grad()
    predicciones = modelo(batch_X)
    loss = criterion(predicciones, batch_y, mode='train')
    loss.backward()

    # max_norm=inf → no recorta nada, solo mide y devuelve la norma real
    norma_actual = torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=float('inf'))
    normas_gradiente.append(norma_actual.item())

    optimizer.step()
    diag_pbar.set_postfix({'norma': f"{norma_actual.item():.2f}"})

# ==========================================
# ESTADÍSTICAS
# ==========================================
normas_gradiente = np.array(normas_gradiente)

print("\n=== ESTADÍSTICAS DE LA NORMA DEL GRADIENTE ===")
print(f"Media:              {normas_gradiente.mean():.4f}")
print(f"Mediana:            {np.median(normas_gradiente):.4f}")
print(f"Desvío estándar:    {normas_gradiente.std():.4f}")
print(f"Percentil 90:       {np.percentile(normas_gradiente, 90):.4f}")
print(f"Percentil 99:       {np.percentile(normas_gradiente, 99):.4f}")
print(f"Máximo:             {normas_gradiente.max():.4f}")
print(f"Mínimo:              {normas_gradiente.min():.4f}")

# ==========================================
# GRÁFICO
# ==========================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(normas_gradiente, alpha=0.7)
ax.axhline(np.median(normas_gradiente), color='green', linestyle='--', label=f'Mediana: {np.median(normas_gradiente):.2f}')
ax.axhline(np.percentile(normas_gradiente, 99), color='red', linestyle='--', label=f'P99: {np.percentile(normas_gradiente, 99):.2f}')
ax.set_xlabel('Batch')
ax.set_ylabel('Norma del gradiente')
ax.set_title('Norma del gradiente por batch (sin clipping)')
ax.set_yscale('log')  # log porque probablemente haya picos muy grandes
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Análisis de Importancia de Variables

In [ ]:
def permutation_importance_unet(modelo, val_loader, criterion, nombres_variables=None, n_repeats=3, device='cuda', seed=None, resume=True):
    """
    Calcula la importancia por permutación para una U-Net con entradas 4D [B, C, H, W].
    """
    modelo.eval()
    
    if seed is not None:
        torch.manual_seed(seed)
    
    # 1. Pérdida baseline sin modificar nada (promedio ponderado por tamaño de batch)
    baseline_loss_acumulada = 0.0
    n_total = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            out = modelo(x_batch)
            loss = criterion(out, y_batch, mode='val')
            baseline_loss_acumulada += loss.item() * x_batch.size(0)
            n_total += x_batch.size(0)
            
    baseline_loss = baseline_loss_acumulada / n_total
    print(f"Pérdida Baseline (Val Loss sin permutar): {baseline_loss:.6f}\n")
    
    # Identificar cantidad de canales (variables de entrada)
    primer_x, _ = next(iter(val_loader))
    num_channels = primer_x.shape[1]
    
    if nombres_variables is None:
        nombres_variables = [f"Var_{i}" for i in range(num_channels)]
        
    importancias_raw = {var: [] for var in nombres_variables}
    
    # 2. Bucle de permutación por repetición y por canal
    for rep in range(n_repeats):
        print(f"--- Repetición {rep + 1}/{n_repeats} ---")
        
        for c, var_name in enumerate(nombres_variables):
            loss_acumulada = 0.0
            n_total_perm = 0
            
            with torch.no_grad():
                for x_batch, y_batch in tqdm(val_loader, desc=f"Permutando {var_name}", leave=False):
                    x_batch = x_batch.clone().to(device)
                    y_batch = y_batch.to(device)
                    
                    # Permutar las muestras a lo largo del batch (Dimensión 0) solo para el canal c
                    perm_idx = torch.randperm(x_batch.size(0))
                    x_batch[:, c, :, :] = x_batch[perm_idx, c, :, :]
                    
                    out = modelo(x_batch)
                    loss = criterion(out, y_batch, mode='val')
                    loss_acumulada += loss.item() * x_batch.size(0)
                    n_total_perm += x_batch.size(0)
                    
            mean_perm_loss = loss_acumulada / n_total_perm
            # Importancia = cuánto AUMENTÓ el error al permutar esta variable
            delta_loss = mean_perm_loss - baseline_loss
            importancias_raw[var_name].append(delta_loss)
            
    # 3. Consolidar en DataFrame
    summary_data = []
    for var_name in nombres_variables:
        vals = importancias_raw[var_name]
        summary_data.append({
            'feature': var_name,
            'importance_mean': np.mean(vals),
            'importance_std': np.std(vals)
        })
        
    df_importances = pd.DataFrame(summary_data).sort_values(by='importance_mean', ascending=False)
    return df_importances, baseline_loss


def graficar_importancia(df_importances):
    plt.figure(figsize=(10, 6))
    plt.barh(df_importances['feature'], df_importances['importance_mean'], xerr=df_importances['importance_std'])
    plt.gca().invert_yaxis()
    plt.xlabel("Incremento medio en Val Loss (Permuted Loss - Baseline Loss)")
    plt.title("Permutation Feature Importance (Ext U-Net Comb V2.1)")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Guardar imagen con el nombre del modelo
    filename = f"../images/pfi_plot_{NOMBRE_MODELO}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Gráfico guardado exitosamente como {filename}")
    
    plt.show()

In [ ]:
nombres_wrf = ['MDBZ', 'HIELO', 'CTT']

archivos = sorted(glob.glob(f'../modelos/*{NOMBRE_MODELO}.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo.load_state_dict(checkpoint['best_model_state_dict'])
modelo.eval()

df_imp, base_loss = permutation_importance_unet(
    modelo=modelo,
    val_loader=val_loader,
    criterion=criterion,
    nombres_variables=nombres_wrf,
    n_repeats=100,
    device=device
)

print(df_imp)
graficar_importancia(df_imp)

# Optimización

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna'], capture_output=True, text=True)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

## Etapa A — Búsqueda de Arquitectura 

In [ ]:
import optuna

def objective_arquitectura(trial):
    tiempo_inicio_trial = time.time()

    # ==========================================
    # HIPERPARÁMETROS DE ARQUITECTURA
    # ==========================================
    n_niveles      = trial.suggest_int('n_niveles', 2, 4)
    n_filtros_base = trial.suggest_categorical('n_filtros_base', [16, 32, 64, 128])
    
    dropout_ps = [
        trial.suggest_float(f'dropout_nivel_{i}', 0.0, 0.3)
        for i in range(n_niveles)
    ]

    kernel_sizes = [
        trial.suggest_categorical(f'kernel_nivel_{i}', [3, 5, 7, 9,11])
        for i in range(n_niveles)
    ]
    
    # ==========================================
    # HIPERPARÁMETROS DE ENTRENAMIENTO (FIJOS)
    # ==========================================
    LR             = 1e-5
    BATCH_SIZE     = 16
    MAX_PROB       = 0.75
    PESO_POSITIVO  = 50.0
    MIN_PROB       = 0.0
    EPOCAS_TRIAL   = 120
    PATIENCE_TRIAL = 12
    EPOCAS_PARA_MAX_PROB = 60

    # ==========================================PROB_SLOPE = (MAX_PROB - MIN_PROB) / (EPOCAS_PARA_MAX_PROB * len(train_loader))
    # DATALOADERS
    # ==========================================
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader_trial = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                     shuffle=True, generator=generator,
                                     num_workers=0, pin_memory=True)
    val_loader_trial   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                                     shuffle=False, num_workers=0, pin_memory=True)

    # ==========================================
    # PROB_SLOPE, MODELO, OPTIMIZER, LOSS Y SCHEDULER
    # ==========================================
    pasos_por_epoca = len(train_loader_trial)
    PROB_SLOPE = (MAX_PROB - MIN_PROB) / (pasos_por_epoca * EPOCAS_PARA_MAX_PROB)

    torch.manual_seed(SEED + trial.number)
    modelo_trial = UNet_WRF(
        in_channels=3,
        n_niveles=n_niveles, n_filtros_base=n_filtros_base,
        kernel_sizes=kernel_sizes, dropout_ps=dropout_ps
    ).to(device)

    # n_parametros = sum(p.numel() for p in modelo_trial.parameters())
    # penalizacion_parametros = n_parametros / 1_000_000.0
    
    criterion_trial = HybridFourierLossV2(
        min_prob=MIN_PROB, max_prob=MAX_PROB, prob_slope=PROB_SLOPE,
        peso_positivo=PESO_POSITIVO,
        min_y=train_dataset.min_y, max_y=train_dataset.max_y
    ).to(device)
    
    optimizer_trial = optim.Adam(
        # V2.1 ONLY
        # list(modelo_trial.parameters()) + list(criterion_trial.parameters()),
        modelo_trial.parameters(),
        lr=LR
    )
    
    scheduler_trial = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_trial, T_max=EPOCAS_TRIAL, eta_min=1e-7
    )

    mejor_objetivo    = float('inf')
    epocas_sin_mejora = 0

    # ==========================================
    # BUCLE DE ENTRENAMIENTO DEL TRIAL
    # ==========================================
    try:
        for epoch in range(EPOCAS_TRIAL):
            modelo_trial.train()
            for batch_X, batch_y in train_loader_trial:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_trial.zero_grad()
                pred = modelo_trial(batch_X)
                loss = criterion_trial(pred, batch_y, mode='train')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(modelo_trial.parameters(), max_norm=65.0)
                optimizer_trial.step()

            modelo_trial.eval()
            val_loss_acumulada = 0.0
            with torch.no_grad():
                for batch_X, batch_y in val_loader_trial:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    pred = modelo_trial(batch_X)
                    loss = criterion_trial(pred, batch_y, mode='val')
                    val_loss_acumulada += loss.item() * batch_X.size(0)

            epoch_val_loss = val_loss_acumulada / len(val_loader_trial.dataset)
            # Detectar NaN/Inf y podar el trial en lugar de dejarlo completarse con basura
            if not np.isfinite(epoch_val_loss):
                print(f"[Trial {trial.number}] val_loss no finito ({epoch_val_loss}) — podando trial.")
                raise optuna.TrialPruned()
                
            epoch_objetivo = epoch_val_loss # + penalizacion_parametros
            
            scheduler_trial.step()

            if epoch_objetivo < mejor_objetivo:
                mejor_objetivo = epoch_objetivo
                epocas_sin_mejora = 0
            else:
                epocas_sin_mejora += 1
                if epocas_sin_mejora >= PATIENCE_TRIAL:
                    break

            trial.report(epoch_objetivo, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Epoch {epoch+1}/{EPOCAS_TRIAL} | val_loss={epoch_val_loss:.4f} | objetivo={epoch_objetivo:.4f}")

        trial.set_user_attr('tiempo_trial_seg', time.time() - tiempo_inicio_trial)
        # V2.1 ONLY
        # trial.set_user_attr('log_var_wmse_final', criterion_trial.log_var_wmse.item())
        # trial.set_user_attr('log_var_fal_final', criterion_trial.log_var_fal.item())
        # trial.set_user_attr('log_var_fcl_final', criterion_trial.log_var_fcl.item())
        return mejor_objetivo

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] OOM — se descarta este trial.")
        raise optuna.TrialPruned()

    except Exception as e:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] Error inesperado: {type(e).__name__}: {e} — se descarta este trial.")
        raise optuna.TrialPruned()

In [ ]:
study_arq = optuna.create_study(
    study_name=f'busqueda_arquitectura_{NOMBRE_MODELO}',
    storage=f'sqlite:///../estudios/optuna_arquitectura_{NOMBRE_MODELO}.db',
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=25),
    load_if_exists=True
)

study_arq.optimize(objective_arquitectura, timeout=216*3600, catch=(Exception,))

In [ ]:
study_arq = optuna.load_study(
    study_name=f'busqueda_arquitectura_{NOMBRE_MODELO}',
    storage=f'sqlite:///../estudios/optuna_arquitectura_{NOMBRE_MODELO}.db'
)

print(f"Estudio cargado: {len(study_arq.trials)} trials en total")

In [ ]:
# =====================================================================
# RESUMEN DE LA ETAPA A — BÚSQUEDA DE ARQUITECTURA
# =====================================================================

trials_completados = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.COMPLETE]
trials_podados      = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.PRUNED]
trials_fallidos     = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.FAIL]

print("="*60)
print("RESUMEN ETAPA A — BÚSQUEDA DE ARQUITECTURA")
print("="*60)
print(f"Total de trials registrados:   {len(study_arq.trials)}")
print(f"  -> Completados exitosamente: {len(trials_completados)}")
print(f"  -> Podados (pruned):         {len(trials_podados)}")
print(f"  -> Fallidos:                 {len(trials_fallidos)}")

# --- Detectar cuántos completados terminaron en NaN ---
trials_nan = [t for t in trials_completados if t.value is not None and np.isnan(t.value)]
print(f"  -> Completados con valor NaN: {len(trials_nan)}")

print("\n" + "-"*60)
print("MEJOR TRIAL ENCONTRADO")
print("-"*60)
if len(trials_completados) > 0:
    print(f"Trial número: {study_arq.best_trial.number}")
    print(f"Val Loss:     {study_arq.best_trial.value:.6f}")
    print(f"\nHiperparámetros de arquitectura:")
    for k, v in study_arq.best_trial.params.items():
        print(f"  {k}: {v}")
    # --- Obtención segura de atributos ---
    n_params = study_arq.best_trial.user_attrs.get('n_parametros', 'N/D')
    n_params_fmt = f"{n_params:,}" if isinstance(n_params, (int, float)) else n_params
    
    tiempo_seg = study_arq.best_trial.user_attrs.get('tiempo_trial_seg', 'N/D')
    tiempo_fmt = f"{tiempo_seg/60:.2f} minutos" if isinstance(tiempo_seg, (int, float)) else tiempo_seg
    
    print(f"\nCantidad de parámetros del modelo: {n_params_fmt}")
    print(f"Tiempo del trial: {tiempo_fmt}")
else:
    print("⚠️ Ningún trial se completó con éxito.")

print("\n" + "-"*60)
print("TOP 5 MEJORES TRIALS (por Val Loss)")
print("-"*60)
trials_df = study_arq.trials_dataframe()
trials_df_validos = trials_df[trials_df['state'] == 'COMPLETE'].sort_values('value')
cols_mostrar = ['number', 'value'] + [c for c in trials_df_validos.columns if c.startswith('params_')]
display(trials_df_validos[cols_mostrar].head(5))

from optuna.importance import FanovaImportanceEvaluator
evaluador_fijo = FanovaImportanceEvaluator(seed=SEED)

print("\n" + "-"*60)
print("IMPORTANCIA DE HIPERPARÁMETROS")
print("-"*60)
try:
    importancias = optuna.importance.get_param_importances(study_arq, evaluator=evaluador_fijo)
    for k, v in importancias.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"No se pudo calcular importancia: {e}")

print("\n" + "-"*60)
print("TIEMPO TOTAL DE LA ETAPA")
print("-"*60)
tiempos = [t.user_attrs.get('tiempo_trial_seg', 0) for t in trials_completados]
if tiempos:
    print(f"Tiempo total acumulado en trials completados: {sum(tiempos)/3600:.2f} horas")
    print(f"Tiempo promedio por trial completado:         {np.mean(tiempos)/60:.1f} minutos")

# --- Gráfico de importancia ---
import optuna.visualization.matplotlib as vis_mpl

vis_mpl.plot_param_importances(study_arq, evaluator=evaluador_fijo)
plt.tight_layout()
plt.show()

# V2.1 ONLY

# --- Coeficientes de las funciones de pérdida ---

# print("\n" + "-"*60)
# print("PESOS DE UNCERTAINTY WEIGHTING (log_var finales)")
# print("-"*60)


# if len(trials_completados) > 0:
    # print("Mejor trial:")
    # print(f"  log_var_wmse: {study_arq.best_trial.user_attrs.get('log_var_wmse_final', 'N/D'):.4f}")
    # print(f"  log_var_fal:  {study_arq.best_trial.user_attrs.get('log_var_fal_final', 'N/D'):.4f}")
    # print(f"  log_var_fcl:  {study_arq.best_trial.user_attrs.get('log_var_fcl_final', 'N/D'):.4f}")

    # Traducimos a "precision" (peso relativo efectivo) para que sea más interpretable
    # precision_wmse = np.exp(-study_arq.best_trial.user_attrs.get('log_var_wmse_final', 0))
    # precision_fal  = np.exp(-study_arq.best_trial.user_attrs.get('log_var_fal_final', 0))
    # precision_fcl  = np.exp(-study_arq.best_trial.user_attrs.get('log_var_fcl_final', 0))
    # print(f"\n  Precisión efectiva (peso relativo, mayor = más influyente):")
    # print(f"    WMSE: {precision_wmse:.6f}")
    # print(f"    FAL:  {precision_fal:.6f}")
    # print(f"    FCL:  {precision_fcl:.6f}")

# --- Distribución sobre TODOS los trials completados, para ver el patrón general ---
# print("\nDistribución sobre todos los trials completados:")
# cols_logvar = ['number', 'value',
               # 'user_attrs_log_var_wmse_final',
               # 'user_attrs_log_var_fal_final',
               # 'user_attrs_log_var_fcl_final']
# cols_logvar_disponibles = [c for c in cols_logvar if c in trials_df.columns]
# display(trials_df[trials_df['state'] == 'COMPLETE'][cols_logvar_disponibles].describe())

## Etapa B — Optimización de Hiperparámetros de Entrenamiento

In [ ]:
ARQ_FINAL = dict(
    n_niveles=3,
    n_filtros_base=32,
    kernel_sizes=[9, 9, 3],
    dropout_ps=[0.08473501730018196, 0.07849710178981281, 0.32307659365394653],
)

In [ ]:
def objective_entrenamiento(trial):
    torch.cuda.empty_cache()
    tiempo_inicio_trial = time.time()

    # ==========================================
    # HIPERPARÁMETROS DE ENTRENAMIENTO A OPTIMIZAR
    # ==========================================
    lr                   = trial.suggest_float('lr', 1e-5, 5e-4, log=True)
    batch_size     = trial.suggest_categorical('batch_size', [4, 8, 16, 32])
    max_prob             = trial.suggest_float('max_prob', 0.5, 0.95)
    epocas_para_max_prob = trial.suggest_int('epocas_para_max_prob', 10, 100)
    peso_positivo        = trial.suggest_float('peso_positivo', 10.0, 50.0)
    max_norm             = trial.suggest_float('max_norm', 50.0, 10000.0, log=True)

    # ==========================================
    # CONFIGURACIÓN FIJA DE LA ETAPA B
    # ==========================================
    MIN_PROB       = 0.0
    EPOCAS_TRIAL   = 150
    PATIENCE_TRIAL = 15

    # ==========================================
    # DATALOADERS
    # ==========================================
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader_trial = DataLoader(train_dataset, batch_size=batch_size,
                                     shuffle=True, generator=generator,
                                     num_workers=0, pin_memory=True)
    val_loader_trial   = DataLoader(val_dataset, batch_size=batch_size,
                                     shuffle=False, num_workers=0, pin_memory=True)

    # ==========================================
    # MODELO — arquitectura FIJA (ganadora de la Etapa A)
    # ==========================================
    pasos_por_epoca = len(train_loader_trial)
    prob_slope = (max_prob - MIN_PROB) / (pasos_por_epoca * epocas_para_max_prob)
    
    modelo_trial = UNet_WRF(
        in_channels=3,
        n_niveles=ARQ_FINAL['n_niveles'],
        n_filtros_base=ARQ_FINAL['n_filtros_base'],
        kernel_sizes=ARQ_FINAL['kernel_sizes'],
        dropout_ps=ARQ_FINAL['dropout_ps'],
    ).to(device)

    criterion_trial = HybridFourierLossV2(
        min_prob=MIN_PROB, max_prob=max_prob, prob_slope=prob_slope,
        peso_positivo=peso_positivo,
        min_y=train_dataset.min_y, max_y=train_dataset.max_y
    ).to(device)
    
    optimizer_trial = optim.Adam(
        # V2.1 ONLY
        # list(modelo_trial.parameters()) + list(criterion_trial.parameters()),
        modelo_trial.parameters(),
        lr=lr
    )

    scheduler_trial = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_trial, T_max=EPOCAS_TRIAL, eta_min=1e-7
    )

    mejor_val_loss    = float('inf')
    epocas_sin_mejora = 0

    # ==========================================
    # BUCLE DE ENTRENAMIENTO DEL TRIAL
    # ==========================================
    try:
        for epoch in range(EPOCAS_TRIAL):
            modelo_trial.train()
            for batch_X, batch_y in train_loader_trial:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_trial.zero_grad()
                pred = modelo_trial(batch_X)
                loss = criterion_trial(pred, batch_y, mode='train')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(modelo_trial.parameters(), max_norm=max_norm)
                optimizer_trial.step()

            modelo_trial.eval()
            val_loss_acumulada = 0.0
            with torch.no_grad():
                for batch_X, batch_y in val_loader_trial:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    pred = modelo_trial(batch_X)
                    loss = criterion_trial(pred, batch_y, mode='val')
                    val_loss_acumulada += loss.item() * batch_X.size(0)

            epoch_val_loss = val_loss_acumulada / len(val_loader_trial.dataset)

            # --- Chequeo de NaN/Inf ---
            if not np.isfinite(epoch_val_loss):
                print(f"[Trial {trial.number}] val_loss no finito ({epoch_val_loss}) — podando trial.")
                raise optuna.TrialPruned()

            scheduler_trial.step()

            if epoch_val_loss < mejor_val_loss:
                mejor_val_loss = epoch_val_loss
                epocas_sin_mejora = 0
            else:
                epocas_sin_mejora += 1
                if epocas_sin_mejora >= PATIENCE_TRIAL:
                    break

            trial.report(epoch_val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Epoch {epoch+1}/{EPOCAS_TRIAL} | val_loss={epoch_val_loss:.4f}")

        trial.set_user_attr('tiempo_trial_seg', time.time() - tiempo_inicio_trial)
        
        # V2.1 ONLY
        # trial.set_user_attr('log_var_wmse_final', criterion_trial.log_var_wmse.item())
        # trial.set_user_attr('log_var_fal_final', criterion_trial.log_var_fal.item())
        # trial.set_user_attr('log_var_fcl_final', criterion_trial.log_var_fcl.item())
        
        return mejor_val_loss

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] OOM — se descarta este trial.")
        raise optuna.TrialPruned()

    except Exception as e:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] Error inesperado: {type(e).__name__}: {e} — se descarta este trial.")
        raise optuna.TrialPruned()

In [ ]:
import os
import shutil

# 1. Definimos las rutas
ruta_red = f'../estudios/optuna_entrenamiento_{NOMBRE_MODELO}.db'
ruta_local = f'/tmp/optuna_entrenamiento_{NOMBRE_MODELO}.db'

# 2. Si ya tenías un estudio previo en tu carpeta de red, lo traemos al disco local para continuarlo
if os.path.exists(ruta_red):
    print("Copiando base de datos de la red al disco local temporal...")
    shutil.copy2(ruta_red, ruta_local)

# 3. Configuramos Optuna para que use la ruta LOCAL
storage_local = f'sqlite:///{ruta_local}'

study_entrenamiento = optuna.create_study(
    study_name=f'optimizacion_entrenamiento_{NOMBRE_MODELO}',
    storage=storage_local,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=25),
    load_if_exists=True
)

print("Iniciando optimización local...")

# 4. Corremos la optimización
try:
    study_entrenamiento.optimize(objective_entrenamiento, timeout=161*3600, catch=(Exception,))
finally:
    # 5. Pase lo que pase (incluso si lo frenas a mano o hay un error), 
    # guardamos los datos de vuelta en tu carpeta de red.
    print("Guardando resultados en la unidad de red...")
    shutil.copy2(ruta_local, ruta_red)
    print("¡Base de datos respaldada con éxito!")

In [ ]:
study_entrenamiento = optuna.load_study(
    study_name=f'optimizacion_entrenamiento_{NOMBRE_MODELO}',
    storage=f'sqlite:///../estudios/optuna_entrenamiento_{NOMBRE_MODELO}.db'
)

print(f"Estudio cargado: {len(study_entrenamiento.trials)} trials en total")

In [ ]:
# =====================================================================
# RESUMEN DE LA ETAPA B — OPTIMIZACIÓN DE HIPERPARÁMETROS DE ENTRENAMIENTO
# =====================================================================

trials_completados = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.COMPLETE]
trials_podados      = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.PRUNED]
trials_fallidos     = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.FAIL]

print("="*60)
print("RESUMEN ETAPA B — OPTIMIZACIÓN DE ENTRENAMIENTO")
print("="*60)
print(f"Total de trials registrados:   {len(study_entrenamiento.trials)}")
print(f"  -> Completados exitosamente: {len(trials_completados)}")
print(f"  -> Podados (pruned):         {len(trials_podados)}")
print(f"  -> Fallidos:                 {len(trials_fallidos)}")

trials_nan = [t for t in trials_completados if t.value is not None and np.isnan(t.value)]
print(f"  -> Completados con valor NaN: {len(trials_nan)}")

print("\n" + "-"*60)
print("MEJOR TRIAL ENCONTRADO")
print("-"*60)
if len(trials_completados) > 0:
    print(f"Trial número: {study_entrenamiento.best_trial.number}")
    print(f"Val Loss:     {study_entrenamiento.best_trial.value:.6f}")
    print(f"\nHiperparámetros de entrenamiento:")
    for k, v in study_entrenamiento.best_trial.params.items():
        print(f"  {k}: {v}")
    print(f"\nTiempo del trial: {study_entrenamiento.best_trial.user_attrs.get('tiempo_trial_seg', 'N/D')}")
else:
    print("⚠️ Ningún trial se completó con éxito.")

print("\n" + "-"*60)
print("TOP 5 MEJORES TRIALS (por Val Loss)")
print("-"*60)
trials_df = study_entrenamiento.trials_dataframe()
trials_df_validos = trials_df[trials_df['state'] == 'COMPLETE'].sort_values('value')
cols_mostrar = ['number', 'value'] + [c for c in trials_df_validos.columns if c.startswith('params_')]
display(trials_df_validos[cols_mostrar].head(5))

from optuna.importance import FanovaImportanceEvaluator
evaluador_fijo = FanovaImportanceEvaluator(seed=SEED)

print("\n" + "-"*60)
print("IMPORTANCIA DE HIPERPARÁMETROS")
print("-"*60)
try:
    importancias_reproducibles = optuna.importance.get_param_importances(
        study_entrenamiento, evaluator=evaluador_fijo
    )
    for k, v in importancias_reproducibles.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"No se pudo calcular importancia (default): {e}")

print("\n" + "-"*60)
print("TIEMPO TOTAL DE LA ETAPA")
print("-"*60)
tiempos = [t.user_attrs.get('tiempo_trial_seg', 0) for t in trials_completados]
if tiempos:
    print(f"Tiempo total acumulado en trials completados: {sum(tiempos)/3600:.2f} horas")
    print(f"Tiempo promedio por trial completado:         {np.mean(tiempos)/60:.1f} minutos")

# --- Gráfico de importancia ---
import optuna.visualization.matplotlib as vis_mpl

vis_mpl.plot_param_importances(study_entrenamiento, evaluator=evaluador_fijo)
plt.tight_layout()
plt.show()

# V2.1 ONLY

# --- Coeficientes de las funciones de pérdida ---

# print("\n" + "-"*60)
# print("PESOS DE UNCERTAINTY WEIGHTING (log_var finales)")
# print("-"*60)

# if len(trials_completados) > 0:
    # print("Mejor trial:")
    # print(f"  log_var_wmse: {study_arq.best_trial.user_attrs.get('log_var_wmse_final', 'N/D'):.4f}")
    # print(f"  log_var_fal:  {study_arq.best_trial.user_attrs.get('log_var_fal_final', 'N/D'):.4f}")
    # print(f"  log_var_fcl:  {study_arq.best_trial.user_attrs.get('log_var_fcl_final', 'N/D'):.4f}")

    # Traducimos a "precision" (peso relativo efectivo) para que sea más interpretable
    # precision_wmse = np.exp(-study_arq.best_trial.user_attrs.get('log_var_wmse_final', 0))
    # precision_fal  = np.exp(-study_arq.best_trial.user_attrs.get('log_var_fal_final', 0))
    # precision_fcl  = np.exp(-study_arq.best_trial.user_attrs.get('log_var_fcl_final', 0))
    # print(f"\n  Precisión efectiva (peso relativo, mayor = más influyente):")
    # print(f"    WMSE: {precision_wmse:.6f}")
    # print(f"    FAL:  {precision_fal:.6f}")
    # print(f"    FCL:  {precision_fcl:.6f}")

# --- Distribución sobre TODOS los trials completados, para ver el patrón general ---
# print("\nDistribución sobre todos los trials completados:")
# cols_logvar = ['number', 'value',
#                'user_attrs_log_var_wmse_final',
#                'user_attrs_log_var_fal_final',
#                'user_attrs_log_var_fcl_final']
# cols_logvar_disponibles = [c for c in cols_logvar if c in trials_df.columns]
# display(trials_df[trials_df['state'] == 'COMPLETE'][cols_logvar_disponibles].describe())

# Resultados

In [ ]:
def crear_cmap_lightdens(n_colores=256, ancho_transicion=30):
    """
    Colormap basado en 'hot_r', pero con el valor mínimo forzado a blanco
    puro. Los primeros `ancho_transicion` colores interpolan linealmente
    desde blanco hasta el color que 'hot_r' ya tiene en ese punto de la
    rampa, así la transición es continua (sin salto ni tramo plano).
    A partir de ahí, el colormap sigue siendo 'hot_r' sin modificar.
    """
    base = plt.get_cmap('hot_r', n_colores)
    colores = base(np.linspace(0, 1, n_colores))

    for i in range(ancho_transicion):
        t = i / (ancho_transicion - 1)
        colores[i] = (1 - t) * np.array([1, 1, 1, 1]) + t * colores[i]

    return LinearSegmentedColormap.from_list('lightdens_white', colores, N=n_colores)

CMAP_LIGHTDENS = crear_cmap_lightdens()

In [ ]:
def resumen_entrenamiento(checkpoint, nombre_modelo, train_loader, val_loader, batch_size):
    """
    Imprime un resumen del entrenamiento combinando datos del checkpoint
    (guardados durante el entrenamiento) con datos del DataLoader actual
    (que no se guardan en el .pth).
    """
    tiempo_total_seg = sum(checkpoint['historial_tiempo_epoch'])
    horas = int(tiempo_total_seg // 3600)
    minutos = int((tiempo_total_seg % 3600) // 60)
    tiempo_str = f"{horas}h {minutos}m" if horas > 0 else f"{minutos}m"

    print(f"-> Modelo Activo:  {nombre_modelo}")
    print(f"-> Épocas totales: {checkpoint['epochs_entrenadas']}")
    print(f"-> Learning Rate:  {checkpoint['lr']}")
    print(f"-> Batch Size:     {batch_size}")
    print(f"-> Archivos para entrenar (Train): {len(train_loader.dataset)}")
    print(f"-> Archivos para validar (Val): {len(val_loader.dataset)}")
    print(f"-> Batches por época (Train): {len(train_loader)}")
    print(f"-> Tiempo: {tiempo_str}")
    print(f"-> Mejor val_loss: {checkpoint['mejor_val_loss']:.6f} (época {checkpoint['mejor_epoca']})")
    print(f"-> Patience: {checkpoint['patience']}")

def generar_ruta_imagen(carpeta, prefijo, idx=None):
    """
    Genera la ruta de guardado. Si se provee un 'idx', lo incluye en el nombre.
    Si el archivo ya existe, le anexa un contador (_01, _02, etc.) para no pisarlo.
    """
    os.makedirs(carpeta, exist_ok=True)
    
    # Armamos el nombre base. Ej: "reconstruccion_unet_idx_42" o "curva_error_unet"
    if idx is not None:
        nombre_base = f"{prefijo}_idx_{idx}"
    else:
        nombre_base = prefijo
        
    ruta = os.path.join(carpeta, f"{nombre_base}.png")
    
    # Si no existe, usamos ese nombre directamente
    if not os.path.exists(ruta):
        return ruta
        
    # Si ya existe, empezamos a probar con un contador hasta encontrar uno libre
    contador = 1
    while True:
        ruta_con_contador = os.path.join(carpeta, f"{nombre_base}_{contador:02d}.png")
        if not os.path.exists(ruta_con_contador):
            return ruta_con_contador
        contador += 1

def graficar_curva_error(checkpoint, nombre_modelo='Modelo'):
    """
    Grafica la curva de MSE para train y validación a partir de un checkpoint cargado.
    """
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    mejor_epoca    = checkpoint['mejor_epoca']
    mejor_val_loss = checkpoint['mejor_val_loss']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epocas, historial_train, 'o-', label='Train Loss (HFACLV2.1)', color='steelblue')
    ax.plot(epocas, historial_val,   'o-', label='Val Loss (HFACLV2.1)',   color='#722F37')
    
    ax.scatter([mejor_epoca], [mejor_val_loss], s=180, marker='*',
               color='gold', edgecolor='black', zorder=5,
               label=f'Mejor época ({mejor_epoca})')
    ax.axvline(mejor_epoca, color='gray', linestyle='--', alpha=0.5)
    ax.annotate(f'{mejor_val_loss:.4f}',
                xy=(mejor_epoca, mejor_val_loss),
                xytext=(8, 10), textcoords='offset points',
                fontsize=9, fontweight='bold')
    
    ax.set_xlabel('Época')
    ax.set_ylabel('HFACLV2.1 Loss')
    ax.set_title(f'Curva de aprendizaje — {nombre_modelo}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"curva_error_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de error guardada en: {ruta_guardado}")
    
    plt.show()
    
    mejor_epoca = historial_val.index(min(historial_val)) + 1
    print(f"Mejor val_loss: {min(historial_val):.6f} (época {mejor_epoca})")

def graficar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', canal_entrada=0, idx=None):
    VAR_NOMBRES = ['MDBZ', 'HIELO', 'AGUA', 'CTT', 'LPI', 'WMAX']

    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    modelo.eval()
    X, y = dataset[idx]
    with torch.no_grad():
        pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

    y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
    y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())
    entrada = X[canal_entrada].numpy()

    # El real nunca puede superar 150 (clippeado en el dataset) — cualquier
    # exceso es ruido de float. La predicción sí puede superar 150 legítimamente.
    real_max = min(y_real.max(), 150.0)
    pred_max = round(float(y_pred.max()), 1)

    vmax_crudo = max(real_max, pred_max)
    if vmax_crudo <= 0:
        vmax_crudo = 1.0

    # Step "lindo" para los ticks de la colorbar
    paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
    if paso_base >= 10:
        x = int(np.ceil(paso_base / 10.0)) * 10
    elif paso_base > 5:
        x = int(np.ceil(paso_base / 5.0)) * 5
    else:
        x = paso_base
    vmax_real = int(np.ceil(vmax_crudo / x)) * x
    ticks = np.arange(0, vmax_real + x, x)

    # Extensión del dominio y fondo (mismo criterio que la animación)
    ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
    ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

    def dibujar_fondo(ax):
        ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 8, 'color': 'gray'}
        gl.ylabel_style = {'size': 8, 'color': 'gray'}

    fig = plt.figure(figsize=(16, 5), facecolor='white')
    fig.suptitle(f'{nombre_modelo} — Timestep {idx}', fontsize=13)

    ax0 = fig.add_subplot(1, 3, 1, projection=CART_PROJ)
    dibujar_fondo(ax0)
    cf0 = ax0.contourf(LONS, LATS, entrada, transform=crs.PlateCarree(), cmap='Blues')
    ax0.set_title(f'Entrada: {VAR_NOMBRES[canal_entrada]}')
    fig.colorbar(cf0, ax=ax0, fraction=0.046, pad=0.04)

    ax1 = fig.add_subplot(1, 3, 2, projection=CART_PROJ)
    dibujar_fondo(ax1)
    cf1 = ax1.pcolormesh(LONS, LATS, y_real, transform=crs.PlateCarree(),
                          cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax_real, shading='auto')
    ax1.set_title('Real: LIGHTDENS')
    fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04, ticks=ticks)

    ax2 = fig.add_subplot(1, 3, 3, projection=CART_PROJ)
    dibujar_fondo(ax2)
    cf2 = ax2.pcolormesh(LONS, LATS, y_pred, transform=crs.PlateCarree(),
                          cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax_real, shading='auto')
    ax2.set_title(f'Predicción: {nombre_modelo}')
    fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04, ticks=ticks)

    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"reconstruccion_{nombre_limpio}", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Reconstrucción guardada en: {ruta_guardado}")

    plt.show()

    print(f"Real  — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    print(f"Pred  — min: {y_pred.min():.2f}, max: {y_pred.max():.2f}, mean: {y_pred.mean():.4f}")

def agrupar_indices_por_tormenta(dataset):
    """
    Agrupa los índices de `dataset` (TormentaDataset) en tormentas: tramos de
    archivos consecutivos en el tiempo (mismo paso que el resto del split).
    Como el dataset de val solo tiene los samples de las tormentas de
    validación (sin los timesteps de las tormentas de train intercaladas),
    un salto de tiempo entre dos archivos consecutivos marca el límite entre
    una tormenta y la siguiente. No depende de una lista de fechas mantenida
    a mano: si val_tormentas cambia, esto se ajusta solo.

    Devuelve dict id_tormenta -> lista de índices (ordenados), donde
    id_tormenta es la fecha/hora del primer archivo ('YYYY-MM-DD_HHMM').
    """
    n = len(dataset)
    if n == 0:
        return {}
    ts = []
    for idx in range(n):
        m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2})-(\d{2})-(\d{2})', dataset.files[idx].name)
        fecha, h, mi, s = m.group(1), m.group(2), m.group(3), m.group(4)
        ts.append((idx, datetime.strptime(f"{fecha} {h}:{mi}:{s}", "%Y-%m-%d %H:%M:%S")))
    ts.sort(key=lambda x: x[1])
    if n == 1:
        return {f"{ts[0][1]:%Y-%m-%d_%H%M}": [ts[0][0]]}

    paso = min(b[1] - a[1] for a, b in zip(ts[:-1], ts[1:]))
    grupos = {}
    grupo_actual = [ts[0]]
    for a, b in zip(ts[:-1], ts[1:]):
        if b[1] - a[1] == paso:
            grupo_actual.append(b)
        else:
            clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
            grupos[clave] = [i for i, _ in grupo_actual]
            grupo_actual = [b]
    clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
    grupos[clave] = [i for i, _ in grupo_actual]
    return grupos

def animar_reconstruccion_por_tormenta(modelo, dataset, nombre_modelo='Model', fps=5, umbral_ruido=0.5):
    modelo.eval()

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    rutas_generadas = []

    for tormenta_id, indices_tormenta in grupos_por_tormenta.items():
        print(f"\n=== Generating animation for {tormenta_id} ({len(indices_tormenta)} timesteps) ===")

        n_frames = len(indices_tormenta)
        predicciones_cache = []
        reales_cache = []

        for idx in tqdm(indices_tormenta, desc=f"Calculating range — {tormenta_id}"):
            X, y = dataset[idx]
            with torch.no_grad():
                pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

            y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
            y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

            y_pred[y_pred < umbral_ruido] = np.nan
            y_real[y_real < umbral_ruido] = np.nan

            reales_cache.append(y_real)
            predicciones_cache.append(y_pred)

        reales_validos = np.concatenate([r[~np.isnan(r)] for r in reales_cache])
        preds_validos  = np.concatenate([p[~np.isnan(p)] for p in predicciones_cache])

        real_max = min(reales_validos.max(), 150.0) if reales_validos.size > 0 else 0.0
        pred_max = round(float(preds_validos.max()), 1) if preds_validos.size > 0 else 0.0

        vmax_crudo = max(real_max, pred_max)
        if vmax_crudo <= 0:
            vmax_crudo = 10.0

        paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
        if paso_base >= 10:
            x = int(np.ceil(paso_base / 10.0)) * 10
        elif paso_base > 5:
            x = int(np.ceil(paso_base / 5.0)) * 5
        else:
            x = paso_base
        vmax = int(np.ceil(vmax_crudo / x)) * x
        ticks = np.arange(0, vmax + x, x)

        print(f"LIGHTDENS Range ({tormenta_id}): 0 - {vmax} (Step: {x})")

        fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': CART_PROJ})
        fig.suptitle(f'{nombre_modelo} — Validation storm {tormenta_id}', fontsize=15, y=0.98)

        ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
        ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

        def dibujar_fondo(ax):
            ax.set_facecolor('white')
            ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
            ax.add_feature(cfeature.BORDERS, linewidth=0.8)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
            ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 8, 'color': 'gray'}
            gl.ylabel_style = {'size': 8, 'color': 'gray'}

        for ax in axes:
            dibujar_fondo(ax)

        pbar = tqdm(total=n_frames, desc="Rendering Animation")

        def update(frame_idx):
            for ax in axes:
                ax.cla()
                dibujar_fondo(ax)

            cf0 = axes[0].pcolormesh(LONS, LATS, reales_cache[frame_idx], transform=crs.PlateCarree(),
                                      cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax, shading='nearest')
            axes[0].set_title(f'Ground Truth: LIGHTDENS — Frame {frame_idx}')

            cf1 = axes[1].pcolormesh(LONS, LATS, predicciones_cache[frame_idx], transform=crs.PlateCarree(),
                                      cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax, shading='nearest')
            axes[1].set_title(f'Prediction: {nombre_modelo}')

            pbar.update(1)
            return cf0, cf1

        cf_inicial_real, cf_inicial_pred = update(0)

        fig.colorbar(cf_inicial_real, ax=axes[0], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=ticks)
        fig.colorbar(cf_inicial_pred, ax=axes[1], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=ticks)

        plt.tight_layout()

        ani = animation.FuncAnimation(fig, update, frames=range(n_frames), interval=200)

        carpeta_destino = "../videos&gifs"
        os.makedirs(carpeta_destino, exist_ok=True)
        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        nombre_archivo_salida = f"reconstruction_{nombre_limpio}_{tormenta_id}.gif"
        ruta_final = os.path.join(carpeta_destino, nombre_archivo_salida)

        ani.save(ruta_final, writer="pillow", fps=fps)
        pbar.close()
        plt.close(fig)

        print(f"Animation saved to {ruta_final}")
        rutas_generadas.append(ruta_final)

    return rutas_generadas
    
def mostrar_historial_tabular(checkpoint):
    """
    Toma el checkpoint de un modelo y muestra una tabla interactiva con 
    el historial de pérdidas (Train y Val) época por época.
    Resalta en verde la época con el menor error de validación.
    """
    # 1. Extraemos las listas del checkpoint
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    
    # 2. Armamos un DataFrame de Pandas
    df = pd.DataFrame({
        'Época': epocas,
        'Train Loss (HFACLV2.2)': historial_train,
        'Val Loss (HFACLV2.2)': historial_val
    })
    
    # 3. Calculamos la diferencia con la época anterior para ver si mejoró
    mejor_hasta_ayer = df['Val Loss (HFACLV2.1)'].shift(1).cummin().fillna(float('inf'))
    df['¿Mejoró?'] = (df['Val Loss (HFACLV2.1)'] < mejor_hasta_ayer).apply(lambda x: '🟢 Sí' if x else '🔴 No')
    df.loc[0, '¿Mejoró?'] = '🟢 Sí'
    
    # 4. Buscamos cuál fue la mejor época absoluta
    mejor_val_loss = min(historial_val)
    
    # 5. Función para pintar la fila ganadora de verde
    def resaltar_mejor_epoca(fila):
        if fila['Val Loss (HFACLV2.1)'] == mejor_val_loss:
            return ['background-color: #d4edda; color: #155724; font-weight: bold'] * len(fila)
        else:
            return [''] * len(fila)
            
    # 6. Aplicamos el formato (dejamos 6 decimales) y mostramos la tabla
    tabla_estilizada = df.style.apply(resaltar_mejor_epoca, axis=1) \
                               .format({'Train Loss (HFACLV2.2)': "{:.6f}", 
                                        'Val Loss (HFACLV2.2)': "{:.6f}"}) \
                               .hide(axis="index")
    
    print("HISTORIAL DE ENTRENAMIENTO TABULAR")
    print("-" * 40)
    display(tabla_estilizada)

    return df

def calcular_data_range_global(dataset):
    """
    Calcula el rango (max - min) de LIGHTDENS real sobre TODO el dataset.
    """
    max_global = float("-inf")
    min_global = float("inf")
    for idx in tqdm(range(len(dataset)), desc="Calculando data_range global"):
        _, y = dataset[idx]
        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        max_global = max(max_global, y_real.max())
        min_global = min(min_global, y_real.min())
    return max_global - min_global

def calcular_metricas_timestep(y_real, y_pred, data_range=None):
    """
    Calcula un set de métricas de verificación comparando la predicción
    contra LIGHTDENS real, para un único timestep. Todas las métricas
    se evalúan solo sobre los píxeles con actividad eléctrica (observada
    o predicha).
    """
    mascara = (y_real > 0) | (y_pred > 0)
    actividad_total = y_real.sum()  # se mide sobre el dominio completo, no filtrado

    if data_range is None:
        data_range = max(y_real.max(), y_pred.max()) - min(y_real.min(), y_pred.min())

    # Mapa de SSIM local, calculado sobre el campo completo
    if data_range > 0:
        _, mapa_ssim = ssim(y_real, y_pred, data_range=data_range, full=True)
    else:
        mapa_ssim = np.ones_like(y_real)

    # Si no hay actividad observada NI predicha, devolvemos NaN para que pandas lo ignore al graficar
    if mascara.sum() == 0:
        return {'mse': np.nan, 'rmse': np.nan, 'mae': np.nan, 'ssim': np.nan,
                'pod': np.nan, 'far': np.nan, 'csi': np.nan,
                'actividad': actividad_total}

    # Filtramos por la máscara
    y_real_m = y_real[mascara]
    y_pred_m = y_pred[mascara]

    # Métricas de error continuo
    error = y_real_m - y_pred_m
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))
    valor_ssim = mapa_ssim[mascara].mean()

    # Cálculo de las 3 Titánides Categóricas (POD, FAR, CSI)
    obs_bin = (y_real_m > 0)
    pred_bin = (y_pred_m > 0)

    hits = (obs_bin & pred_bin).sum()
    misses = (obs_bin & ~pred_bin).sum()
    false_alarms = (~obs_bin & pred_bin).sum()

    pod = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else np.nan

    return {
        'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim,
        'pod': pod, 'far': far, 'csi': csi,
        'actividad': actividad_total
    }

def evaluar_dataset(modelo, dataset, data_range=None):
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    tormenta_de_idx = {idx: tid for tid, idxs in grupos_por_tormenta.items() for idx in idxs}

    for idx in tqdm(range(len(dataset)), desc="Calculando métricas"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        metricas = calcular_metricas_timestep(y_real, y_pred, data_range=data_range)
        metricas['idx'] = idx
        metricas['tormenta'] = tormenta_de_idx[idx]

        registros.append(metricas)

    return pd.DataFrame(registros)

def graficar_metrica(df_metricas, columna, nombre_modelo='Model', ylabel=None, mostrar_actividad=False):
    """
    Plots the evolution of a verification metric across the validation
    timesteps, coloring each validation storm differently and mapping
    storm ids in the legend (English labels).
    """
    # Determinamos qué columnas necesitamos extraer
    columnas_req = ['idx', columna, 'tormenta']
    if mostrar_actividad and 'actividad' in df_metricas.columns:
        columnas_req.append('actividad')
        
    serie = df_metricas[columnas_req].dropna(subset=[columna])

    with plt.style.context('seaborn-v0_8-darkgrid'):
        fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')

        tormentas_unicas = serie['tormenta'].unique()
        colores = plt.cm.tab10(np.linspace(0, 1, len(tormentas_unicas)))

        # NUEVO: Crear eje secundario para la actividad eléctrica (fondo)
        ax2 = None
        if mostrar_actividad and 'actividad' in df_metricas.columns:
            ax2 = ax.twinx()
            ax2.bar(serie['idx'], serie['actividad'], width=1.0, color='gray', alpha=0.25, 
                    label='Electrical Activity (Total Flashes)')
            ax2.set_ylabel('Total Flashes (Ground Truth)', color='gray', fontweight='bold')
            ax2.tick_params(axis='y', labelcolor='gray')
            ax2.grid(False) # Apagamos la grilla secundaria para no ensuciar la principal

        # Graficamos la métrica principal
        for i, tormenta in enumerate(tormentas_unicas):
            grupo = serie[serie['tormenta'] == tormenta]
            ax.plot(
                grupo['idx'],
                grupo[columna],
                'o-',
                color=colores[i],
                markersize=4,
                linewidth=1.2,
                label=f"Storm {tormenta}",
                zorder=3 # Asegura que la línea quede por encima de las barras
            )

        media = serie[columna].mean()
        ax.axhline(
            media,
            color='black',
            linestyle='--',
            alpha=0.8,
            linewidth=1.5,
            label=f'Overall Mean: {media:.3f}',
            zorder=3
        )

        ax.set_xlabel('Validation Timestep')
        ax.set_ylabel(ylabel or columna.upper(), fontweight='bold')
        ax.set_title(f'{ylabel or columna.upper()} — {nombre_modelo}')

        # Combinar leyendas de ambos ejes si ax2 existe
        # (El bbox_to_anchor se corre a 1.15 para no superponerse con los números del eje secundario)
        if ax2:
            lines_1, labels_1 = ax.get_legend_handles_labels()
            lines_2, labels_2 = ax2.get_legend_handles_labels()
            ax.legend(lines_1 + lines_2, labels_1 + labels_2, 
                      bbox_to_anchor=(1.15, 1), loc='upper left', frameon=True, facecolor='white')
        else:
            ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white')
            
        plt.tight_layout()

        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta_guardado = generar_ruta_imagen('../images', f"{columna}_{nombre_limpio}")

        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Plot for {columna.upper()} saved to: {ruta_guardado}")

        plt.show()

    return media

def graficar_mse(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'mse', nombre_modelo, ylabel='MSE')

def graficar_rmse(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'rmse', nombre_modelo, ylabel='RMSE')

def graficar_mae(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'mae', nombre_modelo, ylabel='MAE')

def graficar_ssim(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'ssim', nombre_modelo, ylabel='SSIM', mostrar_actividad=True)

def graficar_pod(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'pod', nombre_modelo, ylabel='Probability of Detection (POD)', mostrar_actividad=True)

def graficar_far(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'far', nombre_modelo, ylabel='False Alarm Ratio (FAR)', mostrar_actividad=True)

def graficar_csi(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'csi', nombre_modelo, ylabel='Critical Success Index (CSI)', mostrar_actividad=True)

def calcular_r2_global(modelo, dataset, nombre_modelo='Model', ignorar_borde=0):
    """
    Calculates R² as a global metric over all active pixels
    in the dataset, for a single model.
    """
    modelo.eval()
    y_real_total = []
    y_pred_total = []

    for idx in tqdm(range(len(dataset)), desc=f"Calculating R² — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        if ignorar_borde > 0:
            y_real = y_real[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]
            y_pred = y_pred[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]

        mascara = (y_real > 0) | (y_pred > 0)
        if mascara.sum() > 0:
            y_real_total.append(y_real[mascara])
            y_pred_total.append(y_pred[mascara])

    y_real_all = np.concatenate(y_real_total)
    y_pred_all = np.concatenate(y_pred_total)

    ss_res = np.sum((y_real_all - y_pred_all) ** 2)
    ss_tot = np.sum((y_real_all - y_real_all.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    edge_label = f" (ignoring {ignorar_borde}px edge)" if ignorar_borde > 0 else " (full image)"
    print(f"R² {nombre_modelo}{edge_label} — over {len(y_real_all):,} active pixels: {r2:.4f}")

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    sufijo_borde = f"_noedge{ignorar_borde}" if ignorar_borde > 0 else "_full"
    ruta_log = f"../modelos/r2_{nombre_limpio}{sufijo_borde}.txt"
    
    with open(ruta_log, 'w', encoding='utf-8') as f:
        f.write(f"Model: {nombre_modelo}\n")
        f.write(f"Edge crop: {ignorar_borde}px\n")
        f.write(f"Active pixels evaluated: {len(y_real_all):,}\n")
        f.write(f"R²: {r2:.6f}\n")
    print(f"Result saved to: {ruta_log}")

    residuos = y_pred_all - y_real_all

    with plt.style.context('seaborn-v0_8-darkgrid'):
        # Añadido facecolor='white' a la figura también para asegurar consistencia
        fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
        ax.hist(residuos, bins=100, color='#722F37', edgecolor='none', alpha=0.85)
        ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Error = 0')
        ax.axvline(residuos.mean(), color='gold', linestyle='-', linewidth=2,
                   label=f'Mean bias: {residuos.mean():+.2f}')
        
        ax.set_xlim(-100, 100)
        
        ax.set_xlabel('Residuals (Predicted − Actual)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Residuals Distribution — {nombre_modelo}{edge_label}')
        ax.legend()
        plt.tight_layout()

        ruta_hist = f"../images/residuals_{nombre_limpio}{sufijo_borde}.png"
        # Facecolor white al exportar
        plt.savefig(ruta_hist, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Histogram saved to: {ruta_hist}")
        plt.show()

    return r2

def graficar_picos_de_error(df_metricas, modelo, dataset, nombre_modelo='Modelo', top_n=3, metrica='mse', canal_entrada=0):
    """
    Encuentra los timesteps (índices) donde el modelo tuvo el mayor error 
    (los picos más altos) y grafica la reconstrucción para entender qué pasó.
    
    Parámetros
    ----------
    df_metricas   : DataFrame devuelto por evaluar_dataset().
    modelo        : modelo PyTorch ya cargado y en eval mode.
    dataset       : TormentaDataset (val_dataset).
    nombre_modelo : str, para el título.
    top_n         : int, cantidad de picos a mostrar (por defecto los 3 peores).
    metrica       : str, la columna del df a usar para buscar los picos ('mse', 'mae', etc).
    canal_entrada : int, índice del canal a mostrar en la gráfica de entrada.
        Percentil de los valores activos (LIGHTDENS > 0) utilizado como
        límite superior de la escala de color en las reconstrucciones.
        Valores menores achican más la escala y resaltan mejor la estructura
        espacial a costa de saturar los picos más intensos.
    """
    # Verificamos que la métrica exista en el DataFrame
    if metrica not in df_metricas.columns:
        raise ValueError(f"La métrica '{metrica}' no se encuentra en el DataFrame.")

    # Calculamos la media para dar un poco de contexto de "qué tan grave" es el pico
    media_metrica = df_metricas[metrica].mean()
    
    # Extraemos las 'top_n' filas con los valores más altos de la métrica
    df_picos = df_metricas.nlargest(top_n, metrica)
    
    print(f"{"="*60}")
    print(f"🔍 ANALIZANDO LOS {top_n} PEORES CASOS SEGÚN {metrica.upper()}")
    print(f"Media global de {metrica.upper()} en este dataset: {media_metrica:.4f}")
    print(f"{"="*60}\n")
    
    # Iteramos sobre esas peores filas
    for ranking, (_, fila) in enumerate(df_picos.iterrows(), start=1):
        idx_pico = int(fila['idx'])
        valor_pico = fila[metrica]
        
        # Calculamos cuántas veces peor es este pico en comparación a la media
        multiplo = valor_pico / media_metrica if media_metrica > 0 else 0
        
        print(f"⚠️ PICO #{ranking} | Timestep: {idx_pico}")
        print(f"-> {metrica.upper()}: {valor_pico:.4f} (El error es {multiplo:.1f} veces la media global)")
        
        # Graficamos el índice extraído
        graficar_reconstruccion(
            modelo=modelo,
            dataset=dataset,
            nombre_modelo=nombre_modelo,
            canal_entrada=canal_entrada,
            idx=idx_pico,
        )
        print("-" * 60)
        print("\n")

def resumen_pesos_loss(criterion):
    """
    Muestra los pesos aprendidos de Uncertainty Weighting al final
    del entrenamiento, convertidos a sigma^2 para interpretación directa.

    Valores más chicos de sigma^2 indican que ese término tuvo mayor
    peso relativo (más "confianza") durante el entrenamiento.
    """
    log_vars = {
        'WMSE': criterion.log_var_wmse.item(),
        'FAL':  criterion.log_var_fal.item(),
        'FCL':  criterion.log_var_fcl.item(),
    }

    print("PESOS APRENDIDOS (Uncertainty Weighting)")
    print("-" * 45)
    for nombre, log_var in log_vars.items():
        sigma2 = np.exp(log_var)
        precision = np.exp(-log_var)
        print(f"  {nombre:5s} — log_var: {log_var:+.4f} | σ²: {sigma2:.4f} | peso (precisión): {precision:.4f}")

    return log_vars

def graficar_distribucion_curvas(modelo, dataset, nombre_modelo='Modelo', bin_w=1.0, sigma=2.0):
    """Curvas de densidad (eje Y log) de Observed vs Predicted, sobre TODOS los píxeles."""
    modelo.eval()
    xmax = float(dataset.desnormalizar_y(torch.tensor(1.0)))   # máximo posible de la escala
    bins = np.arange(0, xmax + bin_w, bin_w)
    centros = 0.5 * (bins[:-1] + bins[1:])
    cnt_real = np.zeros(len(centros), dtype=np.float64)
    cnt_pred = np.zeros(len(centros), dtype=np.float64)
    pn_min, pn_max, pn_sum, n = np.inf, -np.inf, 0.0, 0   # stats de la salida normalizada

    for idx in tqdm(range(len(dataset)), desc=f"Recolectando — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_norm = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy().ravel())
        y_pred = np.round(dataset.desnormalizar_y(pred_norm).numpy().ravel())
        cnt_real += np.histogram(y_real, bins=bins)[0]
        cnt_pred += np.histogram(y_pred, bins=bins)[0]

        pn = pred_norm.numpy().ravel()
        pn_min, pn_max = min(pn_min, pn.min()), max(pn_max, pn.max())
        pn_sum += pn.sum(); n += pn.size

    fig, ax = plt.subplots(figsize=(7, 5))
    for cnt, lbl, c in [(cnt_real, "Observed", "tab:blue"),
                        (cnt_pred, "Predicted", "tab:orange")]:
        if cnt.sum() == 0:
            continue
            
        dens = cnt / (cnt.sum() * bin_w)
        
        # --- APLICAR SUAVIZADO AQUÍ ---
        if sigma > 0:
            dens = gaussian_filter1d(dens, sigma=sigma)
            
        m = dens > 0
        ax.plot(centros[m], dens[m], label=lbl, color=c, lw=1.2)

    ax.set_yscale("log")
    ax.set_xlabel("flashes / columna")
    ax.set_ylabel("Probability Density")
    ax.set_title(f"Lightning distribution — {nombre_modelo}")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    plt.savefig(f"../images/dist_curvas_{nombre_limpio}.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Salida normalizada del modelo — min: {pn_min:.4f}, max: {pn_max:.4f}, media: {pn_sum/n:.4f}")
    return centros, cnt_real, cnt_pred


def graficar_distribucion_curvas_por_tormenta(modelo, dataset, nombre_modelo='Modelo', bin_w=1.0, sigma=2.0):
    """
    Curvas de densidad (eje Y log) de Observed vs Predicted, una por cada
    tormenta de `dataset` (detectadas automáticamente por salto temporal).
    """
    modelo.eval()
    xmax = float(dataset.desnormalizar_y(torch.tensor(1.0)))
    bins = np.arange(0, xmax + bin_w, bin_w)
    centros = 0.5 * (bins[:-1] + bins[1:])

    grupos = agrupar_indices_por_tormenta(dataset)
    resultados = {}

    for tormenta_id, indices in grupos.items():
        cnt_real = np.zeros(len(centros), dtype=np.float64)
        cnt_pred = np.zeros(len(centros), dtype=np.float64)
        pn_min, pn_max, pn_sum, n = np.inf, -np.inf, 0.0, 0

        for idx in tqdm(indices, desc=f"Recolectando — {tormenta_id}"):
            X, y = dataset[idx]
            with torch.no_grad():
                pred_norm = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

            y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy().ravel())
            y_pred = np.round(dataset.desnormalizar_y(pred_norm).numpy().ravel())
            cnt_real += np.histogram(y_real, bins=bins)[0]
            cnt_pred += np.histogram(y_pred, bins=bins)[0]

            pn = pred_norm.numpy().ravel()
            pn_min, pn_max = min(pn_min, pn.min()), max(pn_max, pn.max())
            pn_sum += pn.sum(); n += pn.size

        fig, ax = plt.subplots(figsize=(7, 5))
        for cnt, lbl, c in [(cnt_real, "Observed", "tab:blue"),
                            (cnt_pred, "Predicted", "tab:orange")]:
            if cnt.sum() == 0:
                continue
                
            dens = cnt / (cnt.sum() * bin_w)
            
            # --- APLICAR SUAVIZADO AQUÍ ---
            if sigma > 0:
                dens = gaussian_filter1d(dens, sigma=sigma)
                
            m = dens > 0
            ax.plot(centros[m], dens[m], label=lbl, color=c, lw=1.2)

        ax.set_yscale("log")
        ax.set_xlabel("flashes / columna")
        ax.set_ylabel("Probability Density")
        ax.set_title(f"Lightning distribution — {nombre_modelo} — Storm {tormenta_id}")
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()

        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta = f"../images/dist_curvas_{nombre_limpio}_{tormenta_id}.png"
        plt.savefig(ruta, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"Storm {tormenta_id} — salida normalizada: min={pn_min:.4f}, max={pn_max:.4f}, media={pn_sum/n:.4f}")

        resultados[tormenta_id] = {
            "centros": centros, "cnt_real": cnt_real, "cnt_pred": cnt_pred,
            "pn_min": pn_min, "pn_max": pn_max, "pn_mean": pn_sum / n,
        }

    return resultados

def calcular_matriz_fss_dataset(modelo, dataloader, dataset,
                                umbrales=(1, 5, 15, 30),
                                ventanas=(1, 3, 5, 15)):
    """
    FSS global en GPU, evaluado solo en los puntos con actividad eléctrica
    observada (LIGHTDENS > 0) ya sea por el modelo o por WRF-Elec. Acumula 
    sum((obs_frac - pred_frac)^2) y sum(obs_frac^2 + pred_frac^2) sobre 
    esos puntos y divide recién al final. Devuelve un np.ndarray de forma
    (len(ventanas), len(umbrales)).
    """
    assert all(v % 2 == 1 for v in ventanas), "Las ventanas deben ser impares (padding simétrico)."

    modelo.eval()
    U, V = len(umbrales), len(ventanas)
    umb = torch.tensor(umbrales, device=device, dtype=torch.float32).view(1, U, 1, 1)

    num = torch.zeros(V, U, device=device, dtype=torch.float64)
    den = torch.zeros(V, U, device=device, dtype=torch.float64)

    with torch.no_grad():
        for X, y in tqdm(dataloader, desc="FSS"):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            p = modelo(X)

            y_real = torch.round(dataset.desnormalizar_y(y))
            y_pred = torch.round(dataset.desnormalizar_y(p))

            # Máscara de puntos con actividad eléctrica observada: (B, 1, H, W)
            mask = ((y_real > 0) | (y_pred > 0)).float()

            obs_bin  = (y_real >= umb).float()
            pred_bin = (y_pred >= umb).float()

            for i, v in enumerate(ventanas):
                # Las fracciones se calculan sobre el campo completo (con los vecinos)
                obs_frac  = F.avg_pool2d(obs_bin,  kernel_size=v, stride=1, padding=v // 2)
                pred_frac = F.avg_pool2d(pred_bin, kernel_size=v, stride=1, padding=v // 2)

                # ...pero el error se suma solo donde hay actividad (mask se broadcastea sobre U)
                num[i] += (((obs_frac - pred_frac) ** 2) * mask).sum(dim=(0, 2, 3)).double()
                den[i] += ((obs_frac ** 2 + pred_frac ** 2) * mask).sum(dim=(0, 2, 3)).double()

    fss = torch.where(den == 0, torch.ones_like(den), 1.0 - num / den)
    return fss.cpu().numpy()


def graficar_heatmap_fss(matriz, umbrales=(1, 5, 15, 30), ventanas=(1, 3, 5, 15),
                         nombre_modelo='Modelo'):
    plt.figure(figsize=(7, 5), facecolor='white')
    cmap = sns.diverging_palette(10, 130, as_cmap=True)
    ax = sns.heatmap(
        matriz, annot=True, fmt=".2f", cmap=cmap,
        vmin=0.0, vmax=1.0, center=0.5,
        xticklabels=umbrales, yticklabels=ventanas,
        linewidths=1, linecolor='white',
        cbar_kws={'label': 'Fractional Skill Score (FSS)'}
    )
    ax.invert_yaxis()
    plt.xlabel('Umbral de Intensidad (flashes/columna)', fontweight='bold')
    plt.ylabel('Escala Espacial / Ventana (px)', fontweight='bold')
    plt.title(f'Matriz FSS: Tolerancia Espacial vs Severidad — {nombre_modelo}', pad=15)
    plt.tight_layout()
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    plt.savefig(f"../images/fss_heatmap_{nombre_limpio}.png", dpi=300, bbox_inches='tight')
    plt.show()

def evaluar_ssim_ventanas(modelo, dataset, win_sizes=[3, 5, 7, 11, 15], data_range=None):
    """
    Evalúa el dataset con el modelo actual calculando SSIM para distintas
    ventanas espaciales (win_size). El SSIM se calcula sobre el campo
    completo (para no romper la ventana local) y se promedia solo sobre
    los píxeles con actividad eléctrica (observada o predicha).
    """
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    tormenta_de_idx = {idx: tid for tid, idxs in grupos_por_tormenta.items() for idx in idxs}

    for idx in tqdm(range(len(dataset)), desc="Calculando SSIM multi-ventana"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        mascara = (y_real > 0) | (y_pred > 0)

        registro = {
            'idx': idx,
            'tormenta': tormenta_de_idx[idx],
            'actividad': y_real.sum()
        }

        for w in win_sizes:
            if data_range > 0 and mascara.sum() > 0:
                _, mapa_ssim = ssim(y_real, y_pred, data_range=data_range, win_size=w, full=True)
                registro[f'ssim_{w}x{w}'] = mapa_ssim[mascara].mean()
            elif data_range <= 0:
                registro[f'ssim_{w}x{w}'] = 1.0
            else:
                # Sin actividad ni observada ni predicha: nada que promediar
                registro[f'ssim_{w}x{w}'] = np.nan

        registros.append(registro)

    return pd.DataFrame(registros)

def graficar_sensibilidad_ssim_ventanas(df_resultados, win_sizes=[3, 5, 7, 11, 15], nombre_modelo='Modelo'):
    # Transformamos el DataFrame a formato largo (melt) para seaborn
    columnas_ssim = [f'ssim_{w}x{w}' for w in win_sizes]
    df_long = df_resultados.melt(
        id_vars=['idx', 'tormenta'], 
        value_vars=columnas_ssim,
        var_name='Ventana', 
        value_name='SSIM'
    )

    with plt.style.context('seaborn-v0_8-darkgrid'):
        fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
        
        sns.boxplot(
            data=df_long, 
            x='Ventana', 
            y='SSIM', 
            ax=ax, 
            palette='Blues',
            showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"red"}
        )

        # Imprimir las medias exactas en pantalla
        print("--- Medias de SSIM por ventana ---")
        for col in columnas_ssim:
            print(f"{col}: {df_resultados[col].mean():.4f}")

        ax.set_title(f"Análisis de Sensibilidad de SSIM según el Tamaño de Ventana — {nombre_modelo}", fontweight='bold', fontsize=13)
        ax.set_ylabel("SSIM", fontweight='bold')
        ax.set_xlabel("Tamaño de Ventana (win_size)", fontweight='bold')
        
        plt.tight_layout()
        
        # Limpiamos el nombre del modelo para usarlo en el nombre del archivo
        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta_guardado = f'../images/sensibilidad_ssim_ventanas_{nombre_limpio}.png'
        
        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {ruta_guardado}")
        
        plt.show()

In [ ]:
archivos = sorted(glob.glob(f'../modelos/*{NOMBRE_MODELO}.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo.load_state_dict(checkpoint['best_model_state_dict'])
modelo.eval()
resumen_entrenamiento(checkpoint, NOMBRE_MODELO, train_loader, val_loader, BATCH_SIZE)

In [ ]:
# V2.1 ONLY
# criterion.load_state_dict(checkpoint['criterion_state_dict'])
# resumen_pesos_loss(criterion)

In [ ]:
graficar_curva_error(checkpoint, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_reconstruccion(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, idx=275)

In [ ]:
modelo.eval()
filas = []
with torch.no_grad():
    for i, (X, y) in enumerate(val_loader):
        p = modelo(X.to(device)); y = y.to(device)
        fp = torch.fft.fftn(p.squeeze(1), dim=[-1, -2], norm='ortho')
        fg = torch.fft.fftn(y.squeeze(1), dim=[-1, -2], norm='ortho')
        w = np.sqrt(p.shape[-2] * p.shape[-1])
        filas.append((criterion.wmse(p, y).item(),
                      criterion.fal(fp, fg).item() * w,
                      criterion.fcl(fp, fg).item() * w,
                      p.max().item()))
f = np.array(filas)
for j, n in enumerate(["wmse", "fal*w", "fcl*w", "pred.max"]):
    print(f"{n:9s} mediana={np.median(f[:, j]):10.3f}  max={f[:, j].max():12.3f}")

In [ ]:
modelo.eval()
resultados = []
with torch.no_grad():
    for i, (X, y) in enumerate(val_loader):
        X, y = X.to(device), y.to(device)
        p = modelo(X)

        pred_real = val_dataset.desnormalizar_y(p)
        gt_real   = val_dataset.desnormalizar_y(y)
        pesos = torch.where(gt_real > 0, torch.tensor(criterion.peso_positivo, device=y.device),
                             torch.tensor(1.0, device=y.device))
        wmse_por_sample = ((pred_real - gt_real) ** 2 * pesos).mean(dim=[1, 2, 3])

        for b in range(X.size(0)):
            idx = i * val_loader.batch_size + b   # val_loader tiene shuffle=False
            resultados.append({
                "idx": idx,
                "archivo": val_dataset.files[idx].name,
                "wmse": wmse_por_sample[b].item(),
                "gt_max_real": gt_real[b].max().item(),
                "pred_max_real": pred_real[b].max().item(),
            })

resultados.sort(key=lambda r: -r["wmse"])
for r in resultados[:10]:
    print(f"wmse={r['wmse']:10.2f}  gt_max={r['gt_max_real']:8.3f}  "
          f"pred_max={r['pred_max_real']:8.3f}  archivo={r['archivo']}")

In [ ]:
rutas = animar_reconstruccion_por_tormenta(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint)
ruta_txt = f"../modelos/historial_{NOMBRE_MODELO}.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo, val_dataset, data_range=data_range_val)

In [ ]:
graficar_mse(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_rmse(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_mae(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_pod(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_far(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_csi(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Ventana por default de 7x7
graficar_ssim(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Ventanas a probar (siempre impares)
ventanas_a_probar = [3, 5, 7, 11, 15]

df_ssim_ventanas = evaluar_ssim_ventanas(
    modelo=modelo, 
    dataset=val_dataset, 
    win_sizes=ventanas_a_probar
)

graficar_sensibilidad_ssim_ventanas(df_ssim_ventanas, ventanas_a_probar, NOMBRE_MODELO)

In [ ]:
r2_completo   = calcular_r2_global(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, ignorar_borde=0)
r2_sin_bordes = calcular_r2_global(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, ignorar_borde=10)

print(f"\nDiferencia: {r2_sin_bordes - r2_completo:+.4f}")

In [ ]:
matriz_fss = calcular_matriz_fss_dataset(modelo, val_loader, val_dataset)
graficar_heatmap_fss(matriz_fss, nombre_modelo=NOMBRE_MODELO)

In [ ]:
centros, cnt_real, cnt_pred = graficar_distribucion_curvas(modelo, val_dataset, NOMBRE_MODELO)

In [ ]:
resultados = graficar_distribucion_curvas_por_tormenta(modelo, val_dataset, NOMBRE_MODELO)

In [ ]:
graficar_picos_de_error(df_metricas, modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)